# CKD — Preprocessing
This is the messiest of the 5 datasets: mixed text/number
columns, stray whitespace, and `?` for missing values.
We clean all of that here before encoding and scaling.

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib


## 1. Load raw data

In [20]:
df = pd.read_csv("../../data/raw/ckd.csv")
df.shape


(400, 26)

## 2. Clean up whitespace and '?' across all columns

In [21]:
# Strip whitespace from all string/object columns, replace '?' with NaN
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].astype(str).str.strip().replace("?", np.nan)
        df[col] = df[col].replace("nan", np.nan)


## 3. Convert columns that SHOULD be numeric back to numeric
`pcv`, `wc`, `rc` are lab values but often get loaded as text
because of the messy '?' entries.

In [22]:
should_be_numeric = ["age", "bp", "sg", "al", "su", "bgr", "bu", "sc",
                      "sod", "pot", "hemo", "pcv", "wc", "rc"]

for col in should_be_numeric:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Fill missing numeric values with the column median
for col in should_be_numeric:
    df[col] = df[col].fillna(df[col].median())


## 4. Encode categorical (yes/no, normal/abnormal) columns
Simple binary mapping — these are all two-value columns.

In [23]:
binary_maps = {
    "rbc": {"normal": 1, "abnormal": 0},
    "pc": {"normal": 1, "abnormal": 0},
    "pcc": {"present": 1, "notpresent": 0},
    "ba": {"present": 1, "notpresent": 0},
    "htn": {"yes": 1, "no": 0},
    "dm": {"yes": 1, "no": 0},
    "cad": {"yes": 1, "no": 0},
    "appet": {"good": 1, "poor": 0},
    "pe": {"yes": 1, "no": 0},
    "ane": {"yes": 1, "no": 0},
}

for col, mapping in binary_maps.items():
    df[col] = df[col].map(mapping)
    df[col] = df[col].fillna(df[col].mode()[0])  # fill any leftover missing with most common value


## 5. Encode the target
`classification`: 'ckd' -> 1, 'notckd' -> 0


In [24]:
df["classification"] = df["classification"].map({"ckd": 1, "notckd": 0})
df = df.dropna(subset=["classification"])
df["classification"].value_counts()


classification
1.0    248
0.0    150
Name: count, dtype: int64

## 6. Confirm no missing values remain

In [26]:
print(df.isnull().sum().sum(), "total missing values remaining")


0 total missing values remaining


## 7. Separate features/target, split, and scale

In [25]:
X = df.drop("classification", axis=1)
y = df["classification"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

print("Train:", X_train_scaled.shape, "| Test:", X_test_scaled.shape)


Train: (318, 25) | Test: (80, 25)


## 8. Save cleaned data, scaler, and splits

In [28]:
df.to_csv("../../data/processed/ckd_cleaned.csv", index=False)
joblib.dump(scaler, "../../models/ckd_scaler.pkl")

X_train_scaled.to_csv("../../data/processed/ckd_X_train.csv", index=False)
X_test_scaled.to_csv("../../data/processed/ckd_X_test.csv", index=False)
y_train.to_csv("../../data/processed/ckd_y_train.csv", index=False)
y_test.to_csv("../../data/processed/ckd_y_test.csv", index=False)

print("Saved cleaned data, scaler, and train/test splits.")


Saved cleaned data, scaler, and train/test splits.


## Next step
Open **03_model_training.ipynb**.